<a href="https://colab.research.google.com/github/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution/blob/main/notebooks/05_fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-signal fusion evaluation

Manuscript sections 3.2 and 3.7, research questions 2 and 3: does combining the three channels beat each channel on its own?

For every test image and every transform (clean, JPEG 90/70/50/30, WebP, downscaling, recompression) the three channels look at the same image:

- **provenance**: AI images are signed `trainedAlgorithmicMedia`, human images `digitalCreation`; re-encoded files lose the manifest, as on most platforms,
- **watermark**: AI images carry the mark at the configured α (4, about 36 dB PSNR), detected with the threshold from notebook 03,
- **CNN**: the trained, temperature-calibrated classifier from notebook 02.

The fusion rule uses a valid credential first, then a detected watermark, then the CNN, and abstains when the CNN is unsure (probability between 0.35 and 0.65).
Each single channel is also scored alone: provenance and watermark abstain when they have nothing to say, the CNN always decides.

Before starting: notebook 01 (data), 02 (trained models) and 03 (watermark threshold) have saved their outputs to Drive.
**Runtime > Change runtime type > T4 GPU** (a CPU runtime works but is much slower). About 20-40 minutes per model.

## 1. Get the code

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd /content/repo
!pip install -q -e ".[train,provenance,dev]"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (evaluation will be slow)")

## 2. Restore the data and link the earlier results from Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
D = "/content/drive/MyDrive/thesis-data"
ZIP = f"{D}/harmonized.zip"
if not os.path.exists(ZIP):
    raise FileNotFoundError(f"{ZIP} not found: run notebook 01 through step 8 first, it saves the prepared data there")
!unzip -qo $ZIP -d .
for d in ["models", "watermark", "evaluation"]:
    !mkdir -p $D/$d && rm -rf outputs/$d && ln -s $D/$d outputs/$d
ARCHS = ["resnet50", "efficientnet_b0"]
missing = [p for p in ["outputs/watermark/threshold.json"] + [f"outputs/models/{a}/{f}" for a in ARCHS for f in ("best.pt", "calibration.json")]
           if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"missing {missing}: run notebook 03 (watermark) and the ResNet-50 / EfficientNet-B0 section of notebook 02 first")
print("models and watermark threshold found")

## 3. Quick check
20 images per class, written to a scratch folder, to make sure everything runs before the full evaluation.

In [ ]:
!python scripts/evaluate.py --arch resnet50 --max-images 20 --out /content/smoke

## 4. Full evaluation
First the watermark detector's precision P(AI | detected) is measured on the validation images; fusion uses it as the confidence of a watermark-based decision.
Then every test image is run through every transform. Results go to Google Drive (`thesis-data/evaluation/<model>/`).

In [ ]:
!python scripts/evaluate.py --arch resnet50

In [ ]:
!python scripts/evaluate.py --arch efficientnet_b0

## 5. Results
**coverage**: share of images the system gave a decision on. **accuracy / F1 / FPR** are computed on those decided images
(FPR = human art wrongly flagged as AI). **ECE**: gap between stated confidence and actual accuracy (lower is better).
For fusion, **decided by** shows which channel made the call.

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
SYSTEMS = ["provenance_only", "watermark_only", "cnn_only", "multi_signal"]
for arch in ARCHS:
    df = pd.read_csv(f"outputs/evaluation/{arch}/summary.csv")
    info = json.load(open(f"outputs/evaluation/{arch}/summary.json"))
    order = df.condition.unique()
    wc = info["watermark_confidence"]
    print(f"===== {arch}  (watermark P(AI|detected) = {wc['confidence']:.4f}, measured on validation)")
    for metric in ["coverage", "accuracy", "f1", "fpr", "ece"]:
        print(metric)
        display(df.pivot(index="condition", columns="system", values=metric).loc[order, SYSTEMS].round(4))
    print("fusion: share of images decided by each channel")
    display(df[df.system == "multi_signal"].set_index("condition")[["decided_by_provenance", "decided_by_watermark", "decided_by_cnn", "coverage"]].loc[order].round(4))
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
    df.pivot(index="condition", columns="system", values="accuracy").loc[order, SYSTEMS].plot.bar(ax=axes[0], ylim=(0, 1.05), legend=False)
    axes[0].set_title(f"{arch}: accuracy on decided images"); axes[0].set_ylabel("accuracy")
    df.pivot(index="condition", columns="system", values="coverage").loc[order, SYSTEMS].plot.bar(ax=axes[1], ylim=(0, 1.05))
    axes[1].set_title(f"{arch}: coverage"); axes[1].set_ylabel("share decided"); axes[1].legend(title="system", fontsize=8)
    plt.tight_layout(); plt.show()